# AstroCLIMB 04 — frozen-feature baselines and augmentation

This is the direct handoff from notebook 03. It attaches the frozen representation cache, encodes the much smaller Kaggle object set once, creates leakage-safe grouped folds, evaluates E03–E06, fits each configuration on its full training data, and performs inference on `test.csv`. The 72 GB Hugging Face image split is **not** streamed again.

Required Kaggle inputs: the competition data and the private `astroclimb_01`, `astroclimb_02`, and `astroclimb_03` datasets. The complete experiment pipeline is embedded in this notebook; no companion source file is required. Enable a GPU and Internet.

In [1]:
%pip install -q "transformers>=4.51" "catboost>=1.2" joblib scipy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
"""Shared implementation for AstroCLIMB notebook 04.

The public entry points are deliberately small so the notebook remains readable.
All learned inputs are derived from figures and captions. Metadata is used only
to create leakage-safe folds and to remove cross-source overlap during validation.
"""

from __future__ import annotations

import base64
import csv
import gc
import hashlib
import io
import json
import os
import pickle
import sqlite3
import time
from collections import Counter
from dataclasses import asdict, dataclass
from difflib import SequenceMatcher
from pathlib import Path
from typing import Iterable, Sequence

import joblib
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from scipy.fft import dctn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoImageProcessor, AutoModel, AutoProcessor, AutoTokenizer


LABELS = ("same_figure", "same_paper", "related_papers", "unrelated_papers")
MODALITIES = ("text-text", "text-image", "image-image")
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
MODEL_IDS = {
    "specter": "allenai/specter2_base",
    "siglip": "google/siglip2-base-patch16-naflex",
    "dino": "facebook/dinov2-small",
}
PAIR_STAT_NAMES = ("cos", "l1", "l2", "max")
FEATURE_NAMES = tuple(
    [f"specter_{x}" for x in PAIR_STAT_NAMES]
    + [f"siglip_{x}" for x in PAIR_STAT_NAMES]
    + [f"dino_{x}" for x in PAIR_STAT_NAMES]
    + [
        "word_tfidf",
        "char_tfidf",
        "phash_similarity",
        "text_text",
        "text_image",
        "image_image",
        "min_text_length",
        "max_text_length",
    ]
)


@dataclass
class Config:
    seed: int = 2026
    n_folds: int = 5
    iterations: int = 650
    depth: int = 7
    learning_rate: float = 0.04
    text_batch: int = 64
    image_batch: int = 8
    synthetic_small_per_cell: int = 4000
    use_gpu_catboost: bool = True
    output_dir: str = "/kaggle/working/astroclimb_04"
    train_csv: str | None = None
    test_csv: str | None = None
    retrieval_csv: str | None = None
    metadata_index: str | None = None
    train_manifest: str | None = None
    validation_manifest: str | None = None
    representation_cache: str | None = None


def is_image(value: object) -> bool:
    return isinstance(value, str) and value.lstrip().startswith(("iVBOR", "/9j/"))


def object_key(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _find_all(filename: str) -> list[Path]:
    roots = (Path("/kaggle/input"), Path("."), Path("notebooks"))
    return sorted(
        {p for root in roots if root.exists() for p in root.rglob(filename)},
        key=lambda p: (len(str(p)), str(p)),
    )


def _choose(explicit: str | None, filename: str, preferred: str) -> Path:
    if explicit:
        path = Path(explicit)
        if not path.exists():
            raise FileNotFoundError(path)
        return path
    candidates = _find_all(filename)
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename}; attach the required Kaggle Dataset")
    candidates.sort(key=lambda p: (preferred not in str(p).lower(), len(str(p))))
    return candidates[0]


def _choose_competition_csv(explicit: str | None, filename: str) -> Path:
    if explicit:
        return Path(explicit)
    candidates = _find_all(filename)
    candidates = [p for p in candidates if "astroclimb_0" not in str(p).lower()]
    if not candidates:
        raise FileNotFoundError(f"Could not find competition {filename}")
    if filename == "train.csv":
        candidates = [p for p in candidates if "train_1000" not in p.name] or candidates
    return max(candidates, key=lambda p: p.stat().st_size)


def resolve_inputs(config: Config) -> dict[str, Path]:
    paths = {
        "train_csv": _choose_competition_csv(config.train_csv, "train.csv"),
        "test_csv": _choose_competition_csv(config.test_csv, "test.csv"),
        "retrieval_csv": _choose(config.retrieval_csv, "train_retrieval.csv", "astroclimb_01"),
        "metadata_index": _choose(config.metadata_index, "metadata_index.pkl", "astroclimb_01"),
        "train_manifest": _choose(config.train_manifest, "hf_train_multimodal_pairs.csv", "astroclimb_02"),
        "validation_manifest": _choose(
            config.validation_manifest, "hf_validation_multimodal_pairs.csv", "astroclimb_02"
        ),
    }
    if config.representation_cache:
        cache = Path(config.representation_cache)
    else:
        summaries = _find_all("representation_cache_summary.json")
        if not summaries:
            raise FileNotFoundError("Attach the private astroclimb_03 cache Dataset")
        summaries.sort(key=lambda p: ("astroclimb_03" not in str(p).lower(), len(str(p))))
        cache = summaries[0].parent
    paths["representation_cache"] = cache
    missing = [str(p) for p in paths.values() if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing inputs: " + ", ".join(missing))
    return paths


def load_and_validate_inputs(config: Config, paths: dict[str, Path]) -> dict:
    output = Path(config.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    with paths["metadata_index"].open("rb") as handle:
        metadata = pickle.load(handle)
    records = metadata["records"]
    by_row = {int(r["meta_row"]): r for r in records}
    cache_dir = paths["representation_cache"]
    summary = json.loads((cache_dir / "representation_cache_summary.json").read_text())
    expected = summary["source"]
    assert sha256_file(paths["train_manifest"]) == expected["train_manifest_sha256"]
    assert sha256_file(paths["validation_manifest"]) == expected["validation_manifest_sha256"]
    arrays = {
        "specter_text": np.load(cache_dir / "specter_text_f16.npy", mmap_mode="r"),
        "siglip_text": np.load(cache_dir / "siglip_text_f16.npy", mmap_mode="r"),
        "siglip_image": np.load(cache_dir / "siglip_image_f16.npy", mmap_mode="r"),
        "dino_image": np.load(cache_dir / "dino_image_f16.npy", mmap_mode="r"),
        "phash": np.load(cache_dir / "phash_u64.npy", mmap_mode="r"),
        "specter_done": np.load(cache_dir / "specter_text_done.npy", mmap_mode="r"),
        "siglip_done": np.load(cache_dir / "siglip_text_done.npy", mmap_mode="r"),
        "image_done": np.load(cache_dir / "image_done.npy", mmap_mode="r"),
    }
    train_manifest = pd.read_csv(paths["train_manifest"], keep_default_na=False)
    validation_manifest = pd.read_csv(paths["validation_manifest"], keep_default_na=False)
    target_rows = np.unique(
        np.concatenate(
            [
                train_manifest.obj_1_row.values,
                train_manifest.obj_2_row.values,
                validation_manifest.obj_1_row.values,
                validation_manifest.obj_2_row.values,
            ]
        ).astype(np.int64)
    )
    for mask_name in ("specter_done", "siglip_done", "image_done"):
        assert np.asarray(arrays[mask_name][target_rows]).all(), f"Incomplete {mask_name} cache"
    train_dois = set(train_manifest.obj_1_doi) | set(train_manifest.obj_2_doi)
    validation_dois = set(validation_manifest.obj_1_doi) | set(validation_manifest.obj_2_doi)
    assert train_dois.isdisjoint(validation_dois), "HF train/validation DOI leakage"
    train_rows = set(train_manifest.obj_1_row.astype(int)) | set(train_manifest.obj_2_row.astype(int))
    validation_rows = set(validation_manifest.obj_1_row.astype(int)) | set(validation_manifest.obj_2_row.astype(int))
    assert train_rows.isdisjoint(validation_rows), "HF train/validation row leakage"
    assert set(train_manifest.relationship) == set(LABELS)
    assert len(train_manifest) == 160_000 and len(validation_manifest) == 40_000
    expected_train_cells = {(label, modality): 16_000 for label in LABELS for modality in MODALITIES if label != "same_figure" or modality == "text-image"}
    expected_validation_cells = {key: 4_000 for key in expected_train_cells}
    assert train_manifest.groupby(["relationship", "modality"]).size().to_dict() == expected_train_cells
    assert validation_manifest.groupby(["relationship", "modality"]).size().to_dict() == expected_validation_cells
    print("Validated source hashes, cache masks, pair counts, and DOI disjointness")
    return {
        "output": output,
        "metadata": metadata,
        "records": records,
        "by_row": by_row,
        "summary": summary,
        "arrays": arrays,
        "train_manifest": train_manifest,
        "validation_manifest": validation_manifest,
        "train_dois": train_dois,
        "validation_dois": validation_dois,
    }


class VectorStore:
    def __init__(self, path: Path):
        self.db = sqlite3.connect(path)
        self.db.execute(
            "CREATE TABLE IF NOT EXISTS vectors "
            "(kind TEXT, key TEXT, dim INTEGER, value BLOB, PRIMARY KEY(kind,key))"
        )
        self.db.execute(
            "CREATE TABLE IF NOT EXISTS hashes "
            "(key TEXT PRIMARY KEY, value TEXT NOT NULL)"
        )

    def get(self, kind: str, key: str) -> np.ndarray | None:
        row = self.db.execute(
            "SELECT dim,value FROM vectors WHERE kind=? AND key=?", (kind, key)
        ).fetchone()
        return None if row is None else np.frombuffer(row[1], np.float16, count=row[0]).astype(np.float32)

    def put(self, kind: str, key: str, value: np.ndarray) -> None:
        value = np.asarray(value, np.float16)
        self.db.execute(
            "INSERT OR REPLACE INTO vectors VALUES (?,?,?,?)",
            (kind, key, len(value), value.tobytes()),
        )

    def put_hash(self, key: str, value: int) -> None:
        self.db.execute("INSERT OR REPLACE INTO hashes VALUES (?,?)", (key, str(int(value))))

    def get_hash(self, key: str) -> int | None:
        row = self.db.execute("SELECT value FROM hashes WHERE key=?", (key,)).fetchone()
        return None if row is None else int(row[0])

    def commit(self) -> None:
        self.db.commit()


def _decode_image(value: str) -> Image.Image:
    raw = base64.b64decode(value.strip(), validate=False)
    with Image.open(io.BytesIO(raw)) as image:
        image.load()
        return ImageOps.exif_transpose(image).convert("RGB")


def _normalized(tensor) -> np.ndarray:
    if not torch.is_tensor(tensor):
        if hasattr(tensor, "pooler_output") and tensor.pooler_output is not None:
            tensor = tensor.pooler_output
        elif hasattr(tensor, "last_hidden_state"):
            tensor = tensor.last_hidden_state[:, 0]
        elif isinstance(tensor, (tuple, list)):
            tensor = tensor[0]
        else:
            raise TypeError(f"Cannot extract a tensor from {type(tensor)}")
    tensor = tensor.float()
    tensor = tensor / tensor.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    return tensor.cpu().numpy().astype(np.float16)


def _image_phash(image: Image.Image) -> int:
    gray = ImageOps.exif_transpose(image).convert("L").resize((32, 32), Image.Resampling.LANCZOS)
    coeff = dctn(np.asarray(gray, np.float32), type=2, norm="ortho")[:8, :8].ravel()
    bits = coeff > np.median(coeff[1:])
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return value


def encode_kaggle_objects(config: Config, paths: dict[str, Path], output: Path) -> VectorStore:
    train = pd.read_csv(paths["train_csv"], usecols=["obj_1", "obj_2"], keep_default_na=False)
    test = pd.read_csv(paths["test_csv"], usecols=["obj_1", "obj_2"], keep_default_na=False)
    values = pd.unique(pd.concat([train.obj_1, train.obj_2, test.obj_1, test.obj_2], ignore_index=True))
    texts = [str(v) for v in values if not is_image(v)]
    images = [str(v) for v in values if is_image(v)]
    store = VectorStore(output / "kaggle_object_cache.sqlite")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device != "cuda":
        raise RuntimeError("Enable a Kaggle GPU accelerator")
    dtype = torch.float16
    tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS["specter"], token=os.getenv("HF_TOKEN"))
    specter = AutoModel.from_pretrained(MODEL_IDS["specter"], token=os.getenv("HF_TOKEN"), dtype=dtype).eval().to(device)
    processor = AutoProcessor.from_pretrained(MODEL_IDS["siglip"], token=os.getenv("HF_TOKEN"))
    siglip = AutoModel.from_pretrained(MODEL_IDS["siglip"], token=os.getenv("HF_TOKEN"), dtype=dtype).eval().to(device)
    missing_text = [v for v in texts if store.get("specter", object_key(v)) is None or store.get("siglip", object_key(v)) is None]
    print(f"Kaggle unique objects: {len(texts):,} text, {len(images):,} image")
    print("Text objects pending:", len(missing_text))
    for start in range(0, len(missing_text), config.text_batch):
        batch_text = missing_text[start : start + config.text_batch]
        spec_batch = tokenizer(batch_text, padding=True, truncation=True, max_length=512, return_tensors="pt")
        spec_batch = {k: v.to(device) for k, v in spec_batch.items()}
        sig_batch = processor(text=batch_text, padding="max_length", truncation=True, return_tensors="pt")
        sig_batch = {k: v.to(device) for k, v in sig_batch.items()}
        with torch.inference_mode():
            spec_vec = _normalized(specter(**spec_batch).last_hidden_state[:, 0])
            sig_vec = _normalized(siglip.get_text_features(**sig_batch))
        for value, a, b in zip(batch_text, spec_vec, sig_vec):
            key = object_key(value)
            store.put("specter", key, a)
            store.put("siglip", key, b)
        store.commit()
        if start % (config.text_batch * 25) == 0:
            print(f"Text {min(start + len(batch_text), len(missing_text)):,}/{len(missing_text):,}")
    del specter, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    dino_processor = AutoImageProcessor.from_pretrained(MODEL_IDS["dino"], token=os.getenv("HF_TOKEN"), use_fast=False)
    dino = AutoModel.from_pretrained(MODEL_IDS["dino"], token=os.getenv("HF_TOKEN"), dtype=dtype).eval().to(device)
    missing_images = [
        v for v in images
        if store.get("siglip", object_key(v)) is None or store.get("dino", object_key(v)) is None
    ]
    print("Image objects pending:", len(missing_images))
    for start in range(0, len(missing_images), config.image_batch):
        batch_values = missing_images[start : start + config.image_batch]
        decoded = [_decode_image(v) for v in batch_values]
        sig_batch = processor(images=decoded, padding="max_length", max_num_patches=256, return_tensors="pt")
        sig_batch = {k: v.to(device) for k, v in sig_batch.items()}
        din_batch = dino_processor(images=decoded, return_tensors="pt")
        din_batch = {k: v.to(device) for k, v in din_batch.items()}
        with torch.inference_mode():
            sig_vec = _normalized(siglip.get_image_features(**sig_batch))
            din_vec = _normalized(dino(**din_batch).last_hidden_state[:, 0])
        for value, image, a, b in zip(batch_values, decoded, sig_vec, din_vec):
            key = object_key(value)
            store.put("siglip", key, a)
            store.put("dino", key, b)
            store.put_hash(key, _image_phash(image))
        store.commit()
        if start % (config.image_batch * 25) == 0:
            print(f"Images {min(start + len(batch_values), len(missing_images)):,}/{len(missing_images):,}")
    del siglip, dino
    gc.collect()
    torch.cuda.empty_cache()
    return store


def fit_lexical_models(train_manifest: pd.DataFrame, by_row: dict[int, dict]) -> tuple:
    rows = set(train_manifest.loc[train_manifest.obj_1_type.eq("text"), "obj_1_row"].astype(int))
    rows |= set(train_manifest.loc[train_manifest.obj_2_type.eq("text"), "obj_2_row"].astype(int))
    corpus = [str(by_row[row].get("caption") or "") for row in sorted(rows)]
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=60_000, strip_accents="unicode")
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=60_000)
    word.fit(corpus)
    char.fit(corpus)
    print("Lexical vocabularies:", len(word.vocabulary_), len(char.vocabulary_))
    return word, char


def _pair_stats(left: np.ndarray | None, right: np.ndarray | None) -> list[float]:
    if left is None or right is None:
        return [0.0] * 4
    left = np.asarray(left, np.float32)
    right = np.asarray(right, np.float32)
    distance = np.abs(left - right)
    return [float(left @ right), float(distance.mean()), float(np.linalg.norm(distance)), float(distance.max())]


def _tfidf_pair_scores(vectorizer, left: Sequence[str], right: Sequence[str]) -> np.ndarray:
    if not left:
        return np.empty(0, np.float32)
    a = vectorizer.transform(left)
    b = vectorizer.transform(right)
    return np.asarray(a.multiply(b).sum(axis=1)).ravel().astype(np.float32)


def _modality(a_image: bool, b_image: bool) -> str:
    return "image-image" if a_image and b_image else "text-image" if a_image ^ b_image else "text-text"


def build_hf_features(
    frame: pd.DataFrame, arrays: dict, by_row: dict[int, dict], lexical: tuple
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    features = np.zeros((len(frame), len(FEATURE_NAMES)), np.float32)
    text_left, text_right, text_indices = [], [], []
    modalities = frame.modality.astype(str).values
    for i, row in enumerate(frame.itertuples(index=False)):
        a, b = int(row.obj_1_row), int(row.obj_2_row)
        ai, bi = row.obj_1_type == "image", row.obj_2_type == "image"
        spec_a = None if ai else arrays["specter_text"][a]
        spec_b = None if bi else arrays["specter_text"][b]
        sig_a = arrays["siglip_image"][a] if ai else arrays["siglip_text"][a]
        sig_b = arrays["siglip_image"][b] if bi else arrays["siglip_text"][b]
        din_a = arrays["dino_image"][a] if ai else None
        din_b = arrays["dino_image"][b] if bi else None
        values = _pair_stats(spec_a, spec_b) + _pair_stats(sig_a, sig_b) + _pair_stats(din_a, din_b)
        phash = 1.0 - ((int(arrays["phash"][a]) ^ int(arrays["phash"][b])).bit_count() / 64.0) if ai and bi else 0.0
        lengths = sorted(
            [len(str(by_row[x].get("caption") or "")) / 5000.0 for x, image in ((a, ai), (b, bi)) if not image]
        )
        min_len = lengths[0] if lengths else 0.0
        max_len = lengths[-1] if lengths else 0.0
        mod = _modality(ai, bi)
        features[i] = values + [0.0, 0.0, phash, mod == "text-text", mod == "text-image", mod == "image-image", min_len, max_len]
        if not ai and not bi:
            text_indices.append(i)
            text_left.append(str(by_row[a].get("caption") or ""))
            text_right.append(str(by_row[b].get("caption") or ""))
    for column, vectorizer in ((12, lexical[0]), (13, lexical[1])):
        features[np.asarray(text_indices), column] = _tfidf_pair_scores(vectorizer, text_left, text_right)
    y = frame.relationship.map(LABEL_TO_ID).astype(np.int8).values
    return features, y, modalities


def build_kaggle_features(
    frame: pd.DataFrame, store: VectorStore, lexical: tuple
) -> tuple[np.ndarray, np.ndarray | None, np.ndarray]:
    features = np.zeros((len(frame), len(FEATURE_NAMES)), np.float32)
    text_left, text_right, text_indices = [], [], []
    modalities = []
    for i, row in enumerate(frame.itertuples(index=False)):
        a, b = str(row.obj_1), str(row.obj_2)
        ai, bi = is_image(a), is_image(b)
        ka, kb = object_key(a), object_key(b)
        spec_a = None if ai else store.get("specter", ka)
        spec_b = None if bi else store.get("specter", kb)
        sig_a, sig_b = store.get("siglip", ka), store.get("siglip", kb)
        din_a = store.get("dino", ka) if ai else None
        din_b = store.get("dino", kb) if bi else None
        assert sig_a is not None and sig_b is not None
        values = _pair_stats(spec_a, spec_b) + _pair_stats(sig_a, sig_b) + _pair_stats(din_a, din_b)
        phash = 1.0 - ((store.get_hash(ka) ^ store.get_hash(kb)).bit_count() / 64.0) if ai and bi else 0.0
        lengths = sorted([min(len(x), 5000) / 5000.0 for x, image in ((a, ai), (b, bi)) if not image])
        min_len = lengths[0] if lengths else 0.0
        max_len = lengths[-1] if lengths else 0.0
        mod = _modality(ai, bi)
        modalities.append(mod)
        features[i] = values + [0.0, 0.0, phash, mod == "text-text", mod == "text-image", mod == "image-image", min_len, max_len]
        if not ai and not bi:
            text_indices.append(i)
            text_left.append(a)
            text_right.append(b)
    for column, vectorizer in ((12, lexical[0]), (13, lexical[1])):
        features[np.asarray(text_indices), column] = _tfidf_pair_scores(vectorizer, text_left, text_right)
    y = None
    if all(label in frame for label in LABELS):
        y = np.argmax(frame[list(LABELS)].astype(int).values, axis=1).astype(np.int8)
    return features, y, np.asarray(modalities)


class DSU:
    def __init__(self):
        self.parent: dict[str, str] = {}

    def find(self, item: str) -> str:
        self.parent.setdefault(item, item)
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, left: str, right: str) -> None:
        a, b = self.find(left), self.find(right)
        if a != b:
            self.parent[b] = a


def make_connected_groups(train: pd.DataFrame, retrieval: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame]:
    retrieval = retrieval.set_index("id").reindex(train.id).reset_index()
    assert retrieval.id.astype(str).values.tolist() == train.id.astype(str).values.tolist()
    dsu = DSU()
    row_nodes = []
    for row, meta in zip(train.itertuples(index=False), retrieval.itertuples(index=False)):
        nodes = ["obj:" + object_key(str(row.obj_1)), "obj:" + object_key(str(row.obj_2))]
        for doi in (str(getattr(meta, "obj_1_doi", "")), str(getattr(meta, "obj_2_doi", ""))):
            if doi and doi != "nan":
                nodes.append("doi:" + doi.casefold())
        for node in nodes[1:]:
            dsu.union(nodes[0], node)
        row_nodes.append(nodes)
    groups = np.asarray([dsu.find(nodes[0]) for nodes in row_nodes], dtype=object)
    counts = Counter(groups)
    print("Connected groups:", len(counts), "largest rows:", max(counts.values()))
    return groups, retrieval


def make_folds(y: np.ndarray, groups: np.ndarray, config: Config) -> np.ndarray:
    splitter = StratifiedGroupKFold(config.n_folds, shuffle=True, random_state=config.seed)
    folds = np.full(len(y), -1, np.int8)
    for fold, (_, validation) in enumerate(splitter.split(np.zeros(len(y)), y, groups)):
        folds[validation] = fold
    assert np.all(folds >= 0)
    for fold in range(config.n_folds):
        assert set(groups[folds == fold]).isdisjoint(set(groups[folds != fold]))
        print("Fold", fold, "rows", int((folds == fold).sum()), "classes", Counter(y[folds == fold]))
    return folds


def _small_balanced_manifest(frame: pd.DataFrame, per_cell: int) -> pd.DataFrame:
    parts = []
    for _, part in frame.groupby(["relationship", "modality"], sort=True):
        if len(part) < per_cell:
            raise ValueError("Not enough rows for the balanced E05 subset")
        parts.append(part.iloc[:per_cell])
    result = pd.concat(parts, ignore_index=True)
    assert len(result) == per_cell * 10
    return result


def _legalize(probability: np.ndarray, modalities: np.ndarray) -> np.ndarray:
    result = np.asarray(probability, np.float64).copy()
    illegal = modalities != "text-image"
    result[illegal, LABEL_TO_ID["same_figure"]] = 0.0
    result /= result.sum(axis=1, keepdims=True).clip(min=1e-12)
    return result.astype(np.float32)


def _new_model(config: Config, seed: int):
    from catboost import CatBoostClassifier

    kwargs = dict(
        iterations=config.iterations,
        depth=config.depth,
        learning_rate=config.learning_rate,
        loss_function="MultiClass",
        eval_metric="MultiClass",
        auto_class_weights="Balanced",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
    )
    if config.use_gpu_catboost and torch.cuda.is_available():
        kwargs.update(task_type="GPU", devices="0")
    return CatBoostClassifier(**kwargs)


def _fit_predict(
    X_train: np.ndarray,
    y_train: np.ndarray,
    mod_train: np.ndarray,
    X_eval: np.ndarray,
    mod_eval: np.ndarray,
    specialist: bool,
    config: Config,
    seed: int,
) -> np.ndarray:
    output = np.zeros((len(X_eval), len(LABELS)), np.float32)
    if specialist:
        selections = [(modality, mod_train == modality, mod_eval == modality) for modality in MODALITIES]
    else:
        selections = [("global", np.ones(len(y_train), bool), np.ones(len(X_eval), bool))]
    for offset, (_, train_mask, eval_mask) in enumerate(selections):
        if not eval_mask.any():
            continue
        model = _new_model(config, seed + offset)
        model.fit(X_train[train_mask], y_train[train_mask])
        local = model.predict_proba(X_eval[eval_mask])
        for column, class_id in enumerate(np.asarray(model.classes_, dtype=int)):
            output[np.flatnonzero(eval_mask), class_id] = local[:, column]
    return _legalize(output, mod_eval)


def metric_report(y: np.ndarray, probability: np.ndarray, modalities: np.ndarray) -> dict:
    prediction = probability.argmax(axis=1)
    report = {
        "macro_f1": float(f1_score(y, prediction, average="macro", labels=range(len(LABELS)), zero_division=0)),
        "per_class": classification_report(
            y, prediction, labels=range(len(LABELS)), target_names=LABELS, output_dict=True, zero_division=0
        ),
        "confusion_matrix": confusion_matrix(y, prediction, labels=range(len(LABELS))).tolist(),
        "prediction_distribution": {LABELS[i]: int((prediction == i).sum()) for i in range(len(LABELS))},
        "by_modality": {},
        "cell_f1": {},
    }
    for modality in MODALITIES:
        mask = modalities == modality
        legal = range(4) if modality == "text-image" else range(1, 4)
        report["by_modality"][modality] = float(
            f1_score(y[mask], prediction[mask], average="macro", labels=list(legal), zero_division=0)
        )
        for class_id in legal:
            report["cell_f1"][f"{LABELS[class_id]}|{modality}"] = float(
                f1_score(y[mask] == class_id, prediction[mask] == class_id, zero_division=0)
            )
    return report


def _touches(frame: pd.DataFrame, dois: set[str], rows: set[int]) -> np.ndarray:
    return (
        frame.obj_1_doi.astype(str).isin(dois)
        | frame.obj_2_doi.astype(str).isin(dois)
        | frame.obj_1_row.astype(int).isin(rows)
        | frame.obj_2_row.astype(int).isin(rows)
    ).values


def run_experiments(
    config: Config,
    data: dict,
    train: pd.DataFrame,
    X_kaggle: np.ndarray,
    y_kaggle: np.ndarray,
    mod_kaggle: np.ndarray,
    retrieval: pd.DataFrame,
    folds: np.ndarray,
    X_hf_train: np.ndarray,
    y_hf_train: np.ndarray,
    mod_hf_train: np.ndarray,
    X_hf_validation: np.ndarray,
    y_hf_validation: np.ndarray,
    mod_hf_validation: np.ndarray,
    test: pd.DataFrame,
    X_test: np.ndarray,
    mod_test: np.ndarray,
) -> dict:
    output = data["output"]
    hf_train = data["train_manifest"].reset_index(drop=True)
    hf_validation = data["validation_manifest"].reset_index(drop=True)
    small_frame = _small_balanced_manifest(hf_train, config.synthetic_small_per_cell)
    small_indices = small_frame.index.to_numpy() if small_frame.index.is_unique else None
    # groupby/concat preserves original indices; use them to select the matching cached features.
    small_original_indices = np.concatenate(
        [part.index[: config.synthetic_small_per_cell].values for _, part in hf_train.groupby(["relationship", "modality"], sort=True)]
    )
    experiments = {
        "E03": {"specialist": False, "synthetic": 0},
        "E04": {"specialist": True, "synthetic": 0},
        "E05": {"specialist": True, "synthetic": 40_000},
        "E06": {"specialist": True, "synthetic": 160_000},
    }
    result = {}
    retrieval_indexed = retrieval.set_index("id").reindex(train.id)
    for name, settings in experiments.items():
        started = time.perf_counter()
        oof = np.zeros((len(train), len(LABELS)), np.float32)
        for fold in range(config.n_folds):
            validation_mask = folds == fold
            training_mask = ~validation_mask
            X_parts, y_parts, mod_parts = [X_kaggle[training_mask]], [y_kaggle[training_mask]], [mod_kaggle[training_mask]]
            if settings["synthetic"]:
                val_meta = retrieval_indexed.iloc[np.flatnonzero(validation_mask)]
                val_dois = set(val_meta.obj_1_doi.astype(str)) | set(val_meta.obj_2_doi.astype(str))
                val_dois.discard("")
                val_rows = set(pd.to_numeric(val_meta.obj_1_meta_row, errors="coerce").dropna().astype(int))
                val_rows |= set(pd.to_numeric(val_meta.obj_2_meta_row, errors="coerce").dropna().astype(int))
                candidate_indices = small_original_indices if settings["synthetic"] == 40_000 else np.arange(len(hf_train))
                candidate_frame = hf_train.iloc[candidate_indices]
                keep = ~_touches(candidate_frame, val_dois, val_rows)
                chosen = candidate_indices[keep]
                X_parts.append(X_hf_train[chosen])
                y_parts.append(y_hf_train[chosen])
                mod_parts.append(mod_hf_train[chosen])
            probability = _fit_predict(
                np.concatenate(X_parts), np.concatenate(y_parts), np.concatenate(mod_parts),
                X_kaggle[validation_mask], mod_kaggle[validation_mask], settings["specialist"],
                config, config.seed + fold * 20,
            )
            oof[validation_mask] = probability
        kaggle_report = metric_report(y_kaggle, oof, mod_kaggle)

        # Strict unseen-paper evaluation: discard Kaggle rows whose DOI is unknown or belongs to HF validation.
        known = retrieval_indexed.obj_1_doi.astype(str).ne("") & retrieval_indexed.obj_2_doi.astype(str).ne("")
        safe = known & ~retrieval_indexed.obj_1_doi.astype(str).isin(data["validation_dois"])
        safe &= ~retrieval_indexed.obj_2_doi.astype(str).isin(data["validation_dois"])
        X_parts, y_parts, mod_parts = [X_kaggle[safe.values]], [y_kaggle[safe.values]], [mod_kaggle[safe.values]]
        if settings["synthetic"]:
            chosen = small_original_indices if settings["synthetic"] == 40_000 else np.arange(len(hf_train))
            X_parts.append(X_hf_train[chosen])
            y_parts.append(y_hf_train[chosen])
            mod_parts.append(mod_hf_train[chosen])
        hf_probability = _fit_predict(
            np.concatenate(X_parts), np.concatenate(y_parts), np.concatenate(mod_parts),
            X_hf_validation, mod_hf_validation, settings["specialist"], config, config.seed + 999,
        )
        hf_report = metric_report(y_hf_validation, hf_probability, mod_hf_validation)
        validation_elapsed = time.perf_counter() - started

        # Final full-data fit for genuine test.csv inference. These predictions are
        # saved for comparison/ensembling; only validation-selected submissions
        # should be uploaded to Kaggle.
        final_started = time.perf_counter()
        X_parts, y_parts, mod_parts = [X_kaggle], [y_kaggle], [mod_kaggle]
        if settings["synthetic"]:
            chosen = small_original_indices if settings["synthetic"] == 40_000 else np.arange(len(hf_train))
            X_parts.append(X_hf_train[chosen])
            y_parts.append(y_hf_train[chosen])
            mod_parts.append(mod_hf_train[chosen])
        test_probability = _fit_predict(
            np.concatenate(X_parts), np.concatenate(y_parts), np.concatenate(mod_parts),
            X_test, mod_test, settings["specialist"], config, config.seed + 1999,
        )
        test_prediction = test_probability.argmax(axis=1)
        inference_elapsed = time.perf_counter() - final_started
        probability_frame = pd.DataFrame(
            {f"prob_{label}": test_probability[:, class_id] for class_id, label in enumerate(LABELS)}
        )
        probability_frame.insert(0, "predicted_relationship", np.asarray(LABELS)[test_prediction])
        probability_frame.insert(0, "id", test.id.astype(str).values)
        probability_frame.to_csv(output / f"{name.lower()}_test_predictions.csv", index=False)

        elapsed = time.perf_counter() - started
        result[name] = {
            "configuration": settings,
            "kaggle_grouped_oof": kaggle_report,
            "hf_unseen_paper": hf_report,
            "wall_time_seconds": elapsed,
            "validation_time_seconds": validation_elapsed,
            "final_fit_and_test_inference_seconds": inference_elapsed,
            "feature_variant": "20 frozen scalar features; OCR deferred because notebook 03 does not cache OCR",
        }
        np.savez_compressed(
            output / f"{name.lower()}_probabilities.npz",
            kaggle_id=train.id.astype(str).values,
            kaggle_y=y_kaggle,
            kaggle_modality=mod_kaggle,
            kaggle_fold=folds,
            kaggle_oof=oof,
            hf_pair_id=hf_validation.pair_id.astype(str).values,
            hf_y=y_hf_validation,
            hf_modality=mod_hf_validation,
            hf_probability=hf_probability,
            test_id=test.id.astype(str).values,
            test_modality=mod_test,
            test_probability=test_probability,
        )
        with (output / f"{name.lower()}_report.json").open("w") as handle:
            json.dump(result[name], handle, indent=2, sort_keys=True)
        print(
            name,
            "Kaggle OOF", f"{kaggle_report['macro_f1']:.5f}",
            "HF unseen", f"{hf_report['macro_f1']:.5f}",
            "seconds", f"{elapsed:.1f}",
        )
    with (output / "experiment_comparison.json").open("w") as handle:
        json.dump(result, handle, indent=2, sort_keys=True)
    return result


def write_selected_submissions(results: dict, test: pd.DataFrame, output: Path) -> tuple[pd.DataFrame, str]:
    """Write the planned E04 checkpoint and the validation-selected candidate."""
    table = comparison_table(results)
    table["kaggle_rank"] = table.kaggle_grouped_oof_macro_f1.rank(ascending=False, method="min")
    table["hf_rank"] = table.hf_unseen_paper_macro_f1.rank(ascending=False, method="min")
    table["average_rank"] = (table.kaggle_rank + table.hf_rank) / 2.0
    selected = str(table.sort_values(["average_rank", "kaggle_rank", "experiment"]).iloc[0].experiment)

    def write_submission(experiment: str, filename: str) -> None:
        prediction_path = output / f"{experiment.lower()}_test_predictions.csv"
        predicted = pd.read_csv(prediction_path, keep_default_na=False)
        assert predicted.id.astype(str).tolist() == test.id.astype(str).tolist()
        labels = predicted.predicted_relationship.astype(str).values
        submission = pd.DataFrame({"id": test.id.astype(str).values})
        for label in LABELS:
            submission[label] = (labels == label).astype(np.int8)
        assert submission[list(LABELS)].sum(axis=1).eq(1).all()
        submission.to_csv(output / filename, index=False)

    write_submission("E04", "submission_e04_anchor.csv")
    write_submission(selected, "submission_validation_selected.csv")
    table.to_csv(output / "experiment_comparison_ranked.csv", index=False)
    return table, selected


def comparison_table(results: dict) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "experiment": name,
                "kaggle_grouped_oof_macro_f1": value["kaggle_grouped_oof"]["macro_f1"],
                "hf_unseen_paper_macro_f1": value["hf_unseen_paper"]["macro_f1"],
                "wall_time_seconds": value["wall_time_seconds"],
            }
            for name, value in results.items()
        ]
    ).sort_values("experiment")

## Configuration and input gate

Automatic discovery prefers artifacts named `astroclimb_01`, `astroclimb_02`, and `astroclimb_03`. Set an explicit path below only if discovery selects the wrong file. E03–E06 use the primary seed-2026 manifests produced by notebook 02.

In [3]:
config = Config(
    seed=2026,
    n_folds=5,
    iterations=650,
    depth=7,
    learning_rate=0.04,
    use_gpu_catboost=True,
    output_dir='/kaggle/working/astroclimb_04',
    # train_csv=None, test_csv=None, retrieval_csv=None,
    # metadata_index=None, train_manifest=None, validation_manifest=None,
    # representation_cache=None,
)
paths = resolve_inputs(config)
for name, path in paths.items():
    print(f'{name}: {path}')
data = load_and_validate_inputs(config, paths)
print('Output:', data['output'])

train_csv: /kaggle/input/competitions/astroclimb/train.csv
test_csv: /kaggle/input/competitions/astroclimb/test.csv
retrieval_csv: /kaggle/input/datasets/syedmohaiminulhoque/astroclimb-01/astroclimb_01/train_retrieval.csv
metadata_index: /kaggle/input/datasets/syedmohaiminulhoque/astroclimb-01/astroclimb_01/metadata_index.pkl
train_manifest: /kaggle/input/datasets/syedmohaiminulhoque/astroclimb-02/astroclimb_02/hf_train_multimodal_pairs.csv
validation_manifest: /kaggle/input/datasets/syedmohaiminulhoque/astroclimb-02/astroclimb_02/hf_validation_multimodal_pairs.csv
representation_cache: /kaggle/input/datasets/syedmohaiminulhoque/astroclimb-03/astroclimb_03
Validated source hashes, cache masks, pair counts, and DOI disjointness
Output: /kaggle/working/astroclimb_04


## Encode and cache Kaggle objects

This is resumable through `kaggle_object_cache.sqlite`. Model IDs match notebook 03. Metadata is not used by the encoders.

In [4]:
store = encode_kaggle_objects(config, paths, data['output'])

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Image objects pending: 14598
Images 8/14,598
Images 208/14,598
Images 408/14,598
Images 608/14,598
Images 808/14,598
Images 1,008/14,598
Images 1,208/14,598
Images 1,408/14,598
Images 1,608/14,598
Images 1,808/14,598
Images 2,008/14,598
Images 2,208/14,598
Images 2,408/14,598
Images 2,608/14,598
Images 2,808/14,598
Images 3,008/14,598
Images 3,208/14,598
Images 3,408/14,598
Images 3,608/14,598
Images 3,808/14,598
Images 4,008/14,598
Images 4,208/14,598
Images 4,408/14,598
Images 4,608/14,598
Images 4,808/14,598
Images 5,008/14,598
Images 5,208/14,598
Images 5,408/14,598
Images 5,608/14,598
Images 5,808/14,598
Images 6,008/14,598
Images 6,208/14,598
Images 6,408/14,598
Images 6,608/14,598
Images 6,808/14,598
Images 7,008/14,598
Images 7,208/14,598
Images 7,408/14,598
Images 7,608/14,598
Images 7,808/14,598
Images 8,008/14,598
Images 8,208/14,598
Images 8,408/14,598
Images 8,608/14,598
Images 8,808/14,598
Images 9,008/14,598
Images 9,208/14,598
Images 9,408/14,598
Images 9,608/14,598
Ima

## Build symmetric frozen pair features

The feature block contains SPECTER, SigLIP2, and DINOv2 cosine/L1/L2/max summaries, word and character TF-IDF similarities, pHash similarity, modality flags, and symmetric text lengths. The TF-IDF models are fitted only on the DOI-disjoint HF training-caption partition.

Notebook 03 did not cache image OCR. Therefore this first E03–E06 block records an explicit 20-feature `no_ocr_frozen_cache` variant and never gives OCR to only one domain. OCR is deferred to E19/E22 instead of silently substituting metadata captions for image OCR.

In [5]:
feature_path = data['output'] / 'pair_features.npz'
lexical_path = data['output'] / 'lexical_models.joblib'
train = pd.read_csv(paths['train_csv'], keep_default_na=False)
test = pd.read_csv(paths['test_csv'], keep_default_na=False)
train['id'] = train['id'].astype(str)
test['id'] = test['id'].astype(str)

if lexical_path.exists():
    lexical = joblib.load(lexical_path)
else:
    lexical = fit_lexical_models(data['train_manifest'], data['by_row'])
    joblib.dump(lexical, lexical_path)

if feature_path.exists():
    cached = np.load(feature_path, allow_pickle=True)
    X_kaggle, y_kaggle, mod_kaggle = cached['X_kaggle'].astype(np.float32), cached['y_kaggle'], cached['mod_kaggle']
    X_test, mod_test = cached['X_test'].astype(np.float32), cached['mod_test']
    X_hf_train, y_hf_train, mod_hf_train = cached['X_hf_train'].astype(np.float32), cached['y_hf_train'], cached['mod_hf_train']
    X_hf_validation, y_hf_validation, mod_hf_validation = cached['X_hf_validation'].astype(np.float32), cached['y_hf_validation'], cached['mod_hf_validation']
    print('Loaded cached pair features')
else:
    X_kaggle, y_kaggle, mod_kaggle = build_kaggle_features(train, store, lexical)
    X_test, _, mod_test = build_kaggle_features(test, store, lexical)
    X_hf_train, y_hf_train, mod_hf_train = build_hf_features(data['train_manifest'], data['arrays'], data['by_row'], lexical)
    X_hf_validation, y_hf_validation, mod_hf_validation = build_hf_features(data['validation_manifest'], data['arrays'], data['by_row'], lexical)
    np.savez_compressed(
        feature_path,
        X_kaggle=X_kaggle.astype(np.float16), y_kaggle=y_kaggle, mod_kaggle=mod_kaggle,
        X_test=X_test.astype(np.float16), mod_test=mod_test,
        X_hf_train=X_hf_train.astype(np.float16), y_hf_train=y_hf_train, mod_hf_train=mod_hf_train,
        X_hf_validation=X_hf_validation.astype(np.float16), y_hf_validation=y_hf_validation, mod_hf_validation=mod_hf_validation,
        feature_names=np.asarray(FEATURE_NAMES),
    )
print('Features:', FEATURE_NAMES)
print('Kaggle train/test:', X_kaggle.shape, X_test.shape)
print('HF train/validation:', X_hf_train.shape, X_hf_validation.shape)

Lexical vocabularies: 60000 60000
Features: ('specter_cos', 'specter_l1', 'specter_l2', 'specter_max', 'siglip_cos', 'siglip_l1', 'siglip_l2', 'siglip_max', 'dino_cos', 'dino_l1', 'dino_l2', 'dino_max', 'word_tfidf', 'char_tfidf', 'phash_similarity', 'text_text', 'text_image', 'image_image', 'min_text_length', 'max_text_length')
Kaggle train/test: (10000, 20) (10000, 20)
HF train/validation: (160000, 20) (40000, 20)


## Leakage-safe Kaggle grouped folds

Object hashes and uniquely retrieved paper DOIs are joined with union-find. Rows in the same connected component cannot cross folds. Metadata is used only here and later to exclude cross-source validation overlap; it is never included in `X`.

In [6]:
retrieval = pd.read_csv(paths['retrieval_csv'], keep_default_na=False)
retrieval['id'] = retrieval['id'].astype(str)
groups, retrieval_aligned = make_connected_groups(train, retrieval)
folds = make_folds(y_kaggle, groups, config)
fold_frame = pd.DataFrame({'id': train.id, 'component': groups, 'fold': folds})
fold_frame.to_csv(data['output'] / 'kaggle_connected_component_folds.csv', index=False)
print('Saved fixed folds')

Connected groups: 2686 largest rows: 231
Fold 0 rows 2085 classes Counter({np.int8(2): 644, np.int8(3): 626, np.int8(1): 584, np.int8(0): 231})
Fold 1 rows 2146 classes Counter({np.int8(3): 691, np.int8(2): 632, np.int8(1): 605, np.int8(0): 218})
Fold 2 rows 1965 classes Counter({np.int8(1): 629, np.int8(2): 627, np.int8(3): 527, np.int8(0): 182})
Fold 3 rows 1695 classes Counter({np.int8(1): 551, np.int8(3): 511, np.int8(2): 471, np.int8(0): 162})
Fold 4 rows 2109 classes Counter({np.int8(3): 645, np.int8(1): 631, np.int8(2): 626, np.int8(0): 207})
Saved fixed folds


## E03–E06

- **E03:** global CatBoost, Kaggle only
- **E04:** three modality-specialist CatBoost models, Kaggle only
- **E05:** E04 plus a deterministic 40,000-pair synthetic subset (4,000 per valid cell)
- **E06:** E04 plus all 160,000 synthetic training pairs

For Kaggle OOF, synthetic rows touching a validation-fold object or DOI are removed. For HF validation, Kaggle rows with missing DOI resolution or a validation DOI are removed. Raw probabilities and full metric reports are saved after each experiment.

In [7]:
results = run_experiments(
    config=config, data=data, train=train,
    X_kaggle=X_kaggle, y_kaggle=y_kaggle, mod_kaggle=mod_kaggle,
    retrieval=retrieval_aligned, folds=folds,
    X_hf_train=X_hf_train, y_hf_train=y_hf_train, mod_hf_train=mod_hf_train,
    X_hf_validation=X_hf_validation, y_hf_validation=y_hf_validation, mod_hf_validation=mod_hf_validation,
    test=test, X_test=X_test, mod_test=mod_test,
)
comparison = comparison_table(results)
comparison.to_csv(data['output'] / 'experiment_comparison.csv', index=False)
ranked_comparison, selected_experiment = write_selected_submissions(results, test, data['output'])
display(ranked_comparison)
print('Validation-selected test submission:', selected_experiment)

E03 Kaggle OOF 0.41965 HF unseen 0.42019 seconds 35.3
E04 Kaggle OOF 0.42249 HF unseen 0.42575 seconds 89.5
E05 Kaggle OOF 0.43131 HF unseen 0.45069 seconds 95.6
E06 Kaggle OOF 0.43493 HF unseen 0.45555 seconds 102.4


,experiment,kaggle_grouped_oof_macro_f1,hf_unseen_paper_macro_f1,wall_time_seconds,kaggle_rank,hf_rank,average_rank
0,E03,0.419648,0.420186,35.322401,4.0,4.0,4.0
1,E04,0.422489,0.425754,89.531454,3.0,3.0,3.0
2,E05,0.431308,0.450695,95.575650,2.0,2.0,2.0
3,E06,0.434928,0.455553,102.392454,1.0,1.0,1.0


Validation-selected test submission: E06


## Decision gate and handoff

Inspect both validation columns. If E05/E06 fail to improve HF unseen-paper macro-F1, stop and audit pair generation. If HF improves while Kaggle grouped OOF declines, treat that as synthetic-domain overfitting. If augmentation is competitive on both, proceed to E08 and then E10–E12.

The notebook performs real `test.csv` inference for all four configurations and saves raw test probabilities. It creates `submission_e04_anchor.csv` for the planned reproducibility checkpoint and `submission_validation_selected.csv` using average validation rank. Save `/kaggle/working/astroclimb_04` as a private Kaggle Dataset; do not upload all four candidates to the leaderboard.

In [8]:
best_kaggle = comparison.loc[comparison.kaggle_grouped_oof_macro_f1.idxmax(), 'experiment']
best_hf = comparison.loc[comparison.hf_unseen_paper_macro_f1.idxmax(), 'experiment']
e04 = comparison.set_index('experiment').loc['E04']
e06 = comparison.set_index('experiment').loc['E06']
decision = {
    'best_kaggle_grouped_oof': best_kaggle,
    'best_hf_unseen_paper': best_hf,
    'e06_minus_e04_kaggle': float(e06.kaggle_grouped_oof_macro_f1 - e04.kaggle_grouped_oof_macro_f1),
    'e06_minus_e04_hf': float(e06.hf_unseen_paper_macro_f1 - e04.hf_unseen_paper_macro_f1),
}
decision['next_action'] = (
    'STOP_AND_AUDIT_SYNTHETIC_PAIRS' if decision['e06_minus_e04_hf'] <= 0
    else 'AUDIT_DOMAIN_SHIFT' if decision['e06_minus_e04_kaggle'] < 0
    else 'PROCEED_TO_E08'
)
(data['output'] / 'decision.json').write_text(json.dumps(decision, indent=2, sort_keys=True))
print(json.dumps(decision, indent=2))
print('Artifacts:', sorted(p.name for p in data['output'].iterdir()))

{
  "best_kaggle_grouped_oof": "E06",
  "best_hf_unseen_paper": "E06",
  "e06_minus_e04_kaggle": 0.012438973890016025,
  "e06_minus_e04_hf": 0.029798743218549195,
  "next_action": "PROCEED_TO_E08"
}
Artifacts: ['decision.json', 'e03_probabilities.npz', 'e03_report.json', 'e03_test_predictions.csv', 'e04_probabilities.npz', 'e04_report.json', 'e04_test_predictions.csv', 'e05_probabilities.npz', 'e05_report.json', 'e05_test_predictions.csv', 'e06_probabilities.npz', 'e06_report.json', 'e06_test_predictions.csv', 'experiment_comparison.csv', 'experiment_comparison.json', 'experiment_comparison_ranked.csv', 'kaggle_connected_component_folds.csv', 'kaggle_object_cache.sqlite', 'lexical_models.joblib', 'pair_features.npz', 'submission_e04_anchor.csv', 'submission_validation_selected.csv']
